# NER 第 3 步：标签映射与子词对齐

版本：2026-11-02 preprocessing-v1。**代码准备完成，等待你运行和验收。**

按 **3.1 → 3.6 → 下载 ZIP 和 notebook → 3.7 确认** 执行。CPU 运行时即可。
沿用运行时本地目录，无须 Drive、bucket、Google 授权或 API key。
已有数据会先验证；若运行时已更换，本 notebook 自动重新下载同一固定版本并核对已验收的来源和句序。

本步为 **BERT-base-cased 与 RoBERTa-base** 准备分词器、标签和对齐记录，只下载分词器文件。
不加载模型权重，不训练，不计算测试集预测或 F1。

- IOB2 标签顺序来自数据集元数据；每个原词只有首个子词接受监督，其余子词、特殊符号、补齐位置为 `-100`。
- 最大长度固定为 **256（包含特殊符号）**，作为预先约定的内存设置，不根据测试集调整。
- 超长句按完整词切成不重叠片段，保留所有词；切分只看子词数量，不读取实体边界。
  跨片段的 I- 标签保持原样，后续评估必须先拼回完整原句，不能逐片段计算实体分数。
- 动态右侧补齐；保存原句、原词与片段的对应关系，后续模型输入须排除这些元数据。

产物：两套 Arrow 数据和分词器（运行时本地）、4 张 CSV 表、JSON 检查记录及备份 ZIP。
ZIP 保存代码、来源、配置和检查结果；不含可重新生成的数据与缓存。另用「文件 → 下载 .ipynb」保存本 notebook。

参考：[Hugging Face token classification](https://huggingface.co/docs/transformers/v4.53.3/en/tasks/token_classification)、
[RoBERTa 分词器](https://huggingface.co/docs/transformers/v4.53.3/en/model_doc/roberta)。


## 3.1 检查依赖
沿用前两步的依赖版本，补充分词器版本。若本节安装了依赖，请重启 Python 会话，再从 3.1 开始。
不需要启用 GPU。首次准备需从公开 Hugging Face 下载文件。


In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['USE_TF'] = '0'
import importlib.metadata as metadata
import importlib
import subprocess
import sys

REQUIRED = {'datasets':'4.0.0', 'numpy':'2.0.2', 'pandas':'2.2.2',
            'transformers':'4.53.3', 'tokenizers':'0.21.4', 'huggingface-hub':'0.36.2'}
missing = []
for package, version in REQUIRED.items():
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           *[f'{p}=={v}' for p,v in REQUIRED.items()]])
    raise RuntimeError('Installation finished. Restart the Python session, then rerun 3.1.')
for package, version in REQUIRED.items():
    module = importlib.import_module(package.replace('-', '_'))
    if module.__version__ != version:
        raise RuntimeError('Stale imported package: restart the Python session and rerun 3.1.')
import numpy as np
import pandas as pd
print('Dependencies ready:', REQUIRED)


## 3.2 初始化目录和工具
如恢复到其他目录，只修改 `PROJECT_ROOT`。重新执行完整步骤时，从本节开始产生新的 run_id。
结果写入独立目录；不会覆盖之前的数据文件。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import platform
import random
import uuid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
NOTEBOOK_REVISION = '2026-11-02-preprocessing-v1'
PROJECT_ROOT = Path.home() / 'cs5489_ner_local'
EXPORT_DIR = Path.home() / 'ner_exports'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RESULTS_DIR = PROJECT_ROOT / 'results' / f'step3_{RUN_ID}'
PREPROCESSED_DIR = PROJECT_ROOT / 'data' / f'preprocessed_{RUN_ID}'
STEP3_COMPLETE = DATA_READY = TOKENIZERS_READY = ALIGNMENT_VERIFIED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
for folder in ['config', 'logs', 'src', 'results', 'data']:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
print('Project:', PROJECT_ROOT)
print('Run:', RUN_ID)


In [ ]:
SOURCE_BUNDLE = {'local_backup.py': '"""Verified, portable NER backups. Uses local files and the Python standard library."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nFORMAT = \'cs5489-ner-local-backup-v1\'\nALLOWED_FOLDERS = {\'config\', \'logs\', \'src\', \'results\', \'figures\', \'predictions\', \'notebooks\'}\nALLOWED_ROOT_FILES = {\'README.md\', \'requirements.txt\', \'requirements-step0.txt\'}\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _relative_path(value):\n    if not isinstance(value, str) or not value or \'\\\\\' in value or \':\' in value:\n        raise ValueError(\'Invalid backup member path.\')\n    parts = value.split(\'/\')\n    if any(part in (\'\', \'.\', \'..\') or part.startswith(\'.\') for part in parts):\n        raise ValueError(\'Unsafe backup member path.\')\n    if PurePosixPath(value).is_absolute():\n        raise ValueError(\'Absolute backup member paths are forbidden.\')\n    return PurePosixPath(value)\n\n\ndef _is_project_output(relative, include_checkpoints):\n    parts = relative.parts\n    if any(part.startswith(\'.\') or part == \'__pycache__\' for part in parts):\n        return False\n    filename = relative.name.lower()\n    if (\'credential\' in filename or \'oauth\' in filename or filename.startswith(\'client_secret\') or\n            filename in {\'token.json\', \'tokens.json\'} or\n            any(secret in filename for secret in (\'access_token\', \'refresh_token\', \'id_token\')) or\n            filename.endswith((\'.zip\', \'.pyc\', \'.pem\', \'.key\'))):\n        return False\n    if len(parts) == 1:\n        return relative.name in ALLOWED_ROOT_FILES\n    return parts[0] in ALLOWED_FOLDERS or (include_checkpoints and parts[0] == \'checkpoints\')\n\n\ndef verify_backup(archive_file, max_total_bytes=20 * 1024**3):\n    """Check structure, file sizes and hashes before accepting a local ZIP.\n\n    Hashes detect corruption; they do not establish who created an archive.\n    Restore only backups from a source you trust.\n    """\n    with zipfile.ZipFile(archive_file) as archive:\n        members = archive.infolist()\n        names = [item.filename for item in members]\n        if len(names) != len(set(names)) or \'manifest.json\' not in names:\n            raise ValueError(\'Backup has duplicate members or no manifest.\')\n        if sum(item.file_size for item in members) > max_total_bytes:\n            raise ValueError(\'Backup exceeds the configured uncompressed-size limit.\')\n        for item in members:\n            _relative_path(item.filename)\n            if stat.S_ISLNK(item.external_attr >> 16) or item.is_dir():\n                raise ValueError(\'Links and directory entries are not valid backup members.\')\n        if archive.getinfo(\'manifest.json\').file_size > 10 * 1024**2:\n            raise ValueError(\'Backup manifest is too large.\')\n        manifest = json.loads(archive.read(\'manifest.json\'))\n        if manifest.get(\'format\') != FORMAT or not isinstance(manifest.get(\'files\'), list):\n            raise ValueError(\'Unknown backup format.\')\n        expected_names = {\'manifest.json\'}\n        paths = set()\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            if relative.as_posix() in paths:\n                raise ValueError(\'Duplicate path in backup manifest.\')\n            paths.add(relative.as_posix())\n            member_name = \'project/\' + relative.as_posix()\n            expected_names.add(member_name)\n            if member_name not in names:\n                raise ValueError(\'A manifest file is missing from the ZIP.\')\n            info = archive.getinfo(member_name)\n            if info.file_size != record[\'size_bytes\']:\n                raise ValueError(f\'Backup size check failed: {relative}\')\n            digest = hashlib.sha256()\n            with archive.open(info) as stream:\n                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n                    digest.update(chunk)\n            if digest.hexdigest() != record[\'sha256\']:\n                raise ValueError(f\'Backup checksum check failed: {relative}\')\n        if set(names) != expected_names:\n            raise ValueError(\'Backup contains files not listed in the manifest.\')\n        # Reading every member to EOF above also checks each ZIP CRC.\n    return manifest\n\n\ndef create_backup(project_root, export_dir, step, revision, include_checkpoints=False):\n    """Archive selected project outputs and verify the completed ZIP by reading it."""\n    root = Path(project_root).resolve(strict=True)\n    export = Path(export_dir).resolve()\n    if not root.is_dir() or export.is_relative_to(root):\n        raise ValueError(\'Use an existing project directory and an export directory outside it.\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]+\', step):\n        raise ValueError(\'Use a simple step name, e.g. step0.\')\n    export.mkdir(parents=True, exist_ok=True)\n    records, selected = [], []\n    folders = ALLOWED_FOLDERS | ({\'checkpoints\'} if include_checkpoints else set())\n    candidates = [root / name for name in ALLOWED_ROOT_FILES if (root / name).is_file()]\n    for name in sorted(folders):\n        folder = root / name\n        if folder.is_symlink():\n            raise ValueError(f\'Refusing to back up a linked project directory: {name}\')\n        if folder.is_dir():\n            candidates.extend(folder.rglob(\'*\'))\n    for path in sorted(candidates):\n        relative = path.relative_to(root)\n        if not _is_project_output(relative, include_checkpoints):\n            continue\n        if path.is_symlink() or any((root / Path(*relative.parts[:i])).is_symlink()\n                                    for i in range(1, len(relative.parts))):\n            raise ValueError(f\'Refusing to back up a link: {relative}\')\n        if not path.is_file():\n            continue\n        path.resolve(strict=True).relative_to(root)\n        records.append({\'path\': relative.as_posix(), \'size_bytes\': path.stat().st_size,\n                        \'sha256\': sha256_file(path)})\n        selected.append((path, relative))\n    if not records:\n        raise ValueError(\'No project outputs were selected for backup.\')\n    created = datetime.now(timezone.utc)\n    manifest = {\'format\': FORMAT, \'step\': step, \'revision\': revision,\n                \'created_at_utc\': created.isoformat(), \'files\': records,\n                \'includes_checkpoints\': include_checkpoints}\n    name = f\'NER_{step}_{created:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}.zip\'\n    output = export / name\n    partial = output.with_suffix(\'.partial\')\n    with zipfile.ZipFile(partial, \'x\', compression=zipfile.ZIP_DEFLATED,\n                         compresslevel=2, strict_timestamps=False) as archive:\n        archive.writestr(\'manifest.json\', json.dumps(manifest, indent=2))\n        for path, relative in selected:\n            archive.write(path, arcname=\'project/\' + relative.as_posix())\n    verify_backup(partial)\n    if output.exists():\n        raise FileExistsError(output)\n    partial.rename(output)\n    return {\'path\': str(output), \'name\': output.name, \'size_bytes\': output.stat().st_size,\n            \'sha256\': sha256_file(output), \'file_count\': len(records),\n            \'archive_verified\': True, \'download_confirmed\': False}\n\n\ndef restore_backup(archive_file, destination):\n    """Restore verified bytes into a new directory without executing any code."""\n    manifest = verify_backup(archive_file)\n    target = Path(destination).resolve()\n    if target.exists():\n        raise FileExistsError(\'Choose a new, empty destination name for restoration.\')\n    target.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(archive_file) as archive:\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            path = target.joinpath(*relative.parts)\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(\'project/\' + relative.as_posix()) as source, path.open(\'xb\') as output:\n                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)\n            if sha256_file(path) != record[\'sha256\']:\n                raise ValueError(f\'Restored file checksum failed: {relative}\')\n    return {\'project_root\': str(target), \'step\': manifest[\'step\'], \'file_count\': len(manifest[\'files\'])}\n', 'step1_data.py': '"""Pinned CoNLL-2003 loading and validation. Run real data work in Colab only."""\nimport hashlib\nimport json\nfrom datetime import datetime, timezone\nfrom numbers import Integral\nfrom pathlib import Path\n\nREPO_ID = \'tomaarsen/conll2003\'\nREVISION = \'3a0c0677d3177b6122fceed3239dda7a5de6a514\'\nEXPECTED_ROWS = {\'train\': 14041, \'validation\': 3250, \'test\': 3453}\nTAG_FIELDS = (\'pos_tags\', \'chunk_tags\', \'ner_tags\')\nREQUIRED_FIELDS = (\'id\', \'tokens\', *TAG_FIELDS)\nEXPECTED_NER_LABELS = {\'O\'} | {f\'{p}-{t}\' for p in (\'B\', \'I\') for t in (\'PER\', \'ORG\', \'LOC\', \'MISC\')}\nPARQUET_FILES = {s: f\'data/{s}-00000-of-00001.parquet\' for s in EXPECTED_ROWS}\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.partial\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef load_pinned_dataset(project_root):\n    """Download public data, without auth or remote loading scripts; retain file hashes."""\n    import datasets\n    from huggingface_hub import snapshot_download\n\n    if datasets.__version__ != \'4.0.0\':\n        raise RuntimeError(\'Use datasets==4.0.0; install it and restart the Python session.\')\n    cache = Path(project_root) / \'data\' / \'hf_cache\'\n    snapshot = Path(snapshot_download(\n        repo_id=REPO_ID, repo_type=\'dataset\', revision=REVISION,\n        allow_patterns=[*PARQUET_FILES.values(), \'README.md\'],\n        cache_dir=str(cache / \'hub\'), token=False,\n    ))\n    files = {split: str(snapshot / name) for split, name in PARQUET_FILES.items()}\n    source_files = []\n    for name in [*PARQUET_FILES.values(), \'README.md\']:\n        path = snapshot / name\n        if not path.is_file():\n            raise FileNotFoundError(f\'Pinned source file missing: {name}\')\n        source_files.append({\'path\': name, \'size_bytes\': path.stat().st_size, \'sha256\': file_sha256(path)})\n    dataset = datasets.load_dataset(\'parquet\', data_files=files,\n                                    cache_dir=str(cache / \'arrow\'), token=False)\n    provenance = {\n        \'requested_source\': \'eriktks/conll2003\',\n        \'actual_source\': REPO_ID, \'revision\': REVISION, \'format\': \'parquet\',\n        \'source_url\': f\'https://huggingface.co/datasets/{REPO_ID}/tree/{REVISION}\',\n        \'reason\': \'Development document permits Parquet fallback; datasets 4.0 removes loading scripts.\',\n        \'license_as_stated_by_card\': \'other\',\n        \'source_files\': source_files,\n        \'preprocessing\': \'None: no shuffle, split, filtering, label conversion or token normalization.\',\n        \'parity_scope\': \'Split sizes, metadata, all-row structure/IOB2 and ordered content hashes checked. \'\n                        \'Byte-for-byte equivalence to eriktks/conll2003 has not been established.\',\n        \'recorded_at_utc\': datetime.now(timezone.utc).isoformat(),\n    }\n    return dataset, provenance\n\n\ndef label_names(split):\n    names = {}\n    for field in TAG_FIELDS:\n        feature = split.features[field]\n        values = getattr(getattr(feature, \'feature\', None), \'names\', None)\n        if not values or len(values) != len(set(values)):\n            raise ValueError(f\'{field} must have unique ClassLabel names in feature metadata.\')\n        names[field] = list(values)\n    if set(names[\'ner_tags\']) != EXPECTED_NER_LABELS:\n        raise ValueError(\'NER metadata must contain O and B/I labels for PER, ORG, LOC, MISC.\')\n    return names\n\n\ndef audit_dataset(dataset, expected_rows=None):\n    """Reject corrupt/IOB1 data instead of silently repairing labels or dropping rows.\n\n    expected_rows is injectable for tiny offline fixtures; real runs use EXPECTED_ROWS.\n    Hash input: UTF-8 compact JSON for each core record, decoded tag strings, plus LF.\n    """\n    expected_rows = EXPECTED_ROWS if expected_rows is None else expected_rows\n    if set(dataset) != set(expected_rows):\n        raise ValueError(\'Unexpected dataset splits.\')\n    report = {\'splits\': {}, \'label_names\': None, \'iob2_violations\': 0,\n              \'hash_format\': \'core-fields-decoded-tags-jsonl-utf8-v1\'}\n    for split_name, expected in expected_rows.items():\n        split = dataset[split_name]\n        if len(split) != expected:\n            raise ValueError(f\'{split_name}: expected {expected} rows, received {len(split)}.\')\n        if not set(REQUIRED_FIELDS).issubset(split.column_names):\n            raise ValueError(f\'{split_name}: missing required columns.\')\n        names = label_names(split)\n        if report[\'label_names\'] is None:\n            report[\'label_names\'] = names\n        elif names != report[\'label_names\']:\n            raise ValueError(f\'{split_name}: label metadata differs between splits.\')\n        has_documents = {\'document_id\', \'sentence_id\'}.issubset(split.column_names)\n        digest = hashlib.sha256()\n        tokens_total = 0\n        previous_doc, previous_sentence = None, None\n        document_count = 0\n        for index, row in enumerate(split):\n            where = f\'{split_name}[{index}]\'\n            if row[\'id\'] != str(index):\n                raise ValueError(f\'{where}: id/order changed (expected id={index}).\')\n            tokens = row[\'tokens\']\n            if not isinstance(tokens, list) or not tokens or any(not isinstance(t, str) or not t for t in tokens):\n                raise ValueError(f\'{where}: empty or invalid token list.\')\n            if \'-DOCSTART-\' in tokens:\n                raise ValueError(f\'{where}: unexpected DOCSTART sentinel.\')\n            decoded = {}\n            for field in TAG_FIELDS:\n                tags = row[field]\n                if not isinstance(tags, list) or len(tags) != len(tokens):\n                    raise ValueError(f\'{where}: token/{field} length mismatch.\')\n                if any(isinstance(t, bool) or not isinstance(t, Integral) or not 0 <= t < len(names[field]) for t in tags):\n                    raise ValueError(f\'{where}: invalid {field} index.\')\n                decoded[field] = [names[field][int(t)] for t in tags]\n            previous = \'O\'\n            for position, tag in enumerate(decoded[\'ner_tags\']):\n                if tag.startswith(\'I-\') and previous not in (\'B-\' + tag[2:], tag):\n                    raise ValueError(f\'{where} token {position}: illegal IOB2 transition {previous} -> {tag}.\')\n                previous = tag\n            if has_documents:\n                doc, sentence = row[\'document_id\'], row[\'sentence_id\']\n                if any(isinstance(v, bool) or not isinstance(v, Integral) for v in (doc, sentence)):\n                    raise ValueError(f\'{where}: invalid document/sentence id.\')\n                if previous_doc is None or doc != previous_doc:\n                    if sentence != 0 or (previous_doc is not None and doc != previous_doc + 1):\n                        raise ValueError(f\'{where}: document order is not contiguous.\')\n                    document_count += 1\n                elif sentence != previous_sentence + 1:\n                    raise ValueError(f\'{where}: sentence order is not contiguous.\')\n                previous_doc, previous_sentence = doc, sentence\n            record = {\'id\': row[\'id\'], \'tokens\': tokens, **decoded}\n            digest.update((json.dumps(record, ensure_ascii=False, separators=(\',\', \':\')) + \'\\n\').encode(\'utf-8\'))\n            tokens_total += len(tokens)\n        report[\'splits\'][split_name] = {\n            \'rows\': len(split), \'tokens\': tokens_total, \'ordered_core_sha256\': digest.hexdigest(),\n            \'id_order_verified\': True, \'document_order_verified\': has_documents,\n            \'documents\': document_count if has_documents else None,\n        }\n    return report\n\n\ndef save_and_verify(dataset, project_root, run_id, report):\n    """Keep Arrow data on the runtime; verify persistence without putting data in ZIPs."""\n    from datasets import load_from_disk\n\n    destination = Path(project_root) / \'data\' / f\'conll2003_{run_id}\'\n    if destination.exists():\n        raise FileExistsError(\'This run was already saved. Rerun initialization for a new run_id.\')\n    dataset.save_to_disk(str(destination))\n    restored = load_from_disk(str(destination))\n    check = audit_dataset(restored)\n    if check != report:\n        raise ValueError(\'Saved dataset differs after read-back.\')\n    # Includes auxiliary columns and feature metadata, in addition to ordered core hashes.\n    files = [{\'path\': p.relative_to(destination).as_posix(), \'size_bytes\': p.stat().st_size,\n              \'sha256\': file_sha256(p)} for p in sorted(destination.rglob(\'*\')) if p.is_file()]\n    return {\'path\': str(destination), \'read_back_verified\': True, \'files\': files}\n', 'step2_eda.py': '"""Descriptive NER statistics. No training, model selection or test-set scoring."""\nimport csv\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nSPLITS = (\'train\', \'validation\', \'test\')\nTYPES = (\'PER\', \'ORG\', \'LOC\', \'MISC\')\nPAPER = {\n    \'train\': dict(documents=946, paper_sentences=14987, sentences=14041, tokens=203621,\n                  LOC=7140, MISC=3438, ORG=6321, PER=6600),\n    \'validation\': dict(documents=216, paper_sentences=3466, sentences=3250, tokens=51362,\n                       LOC=1837, MISC=922, ORG=1341, PER=1842),\n    \'test\': dict(documents=231, paper_sentences=3684, sentences=3453, tokens=46435,\n                 LOC=1668, MISC=702, ORG=1661, PER=1617),\n}\nDEFINITIONS = {\n    \'entity\': \'Strict IOB2 span, counted once at B-. End index is exclusive.\',\n    \'surface\': \'Exact tuple of original entity tokens; case-sensitive, no punctuation or whitespace normalization.\',\n    \'unseen_primary\': \'Surface absent among ALL training gold entity types. Mention rate weights occurrences; unique rate weights distinct surfaces within each type.\',\n    \'unseen_typed\': \'Surface absent from training gold entities of the SAME type; reported separately.\',\n    \'casefold\': \'Supplemental case-insensitive comparison uses str.casefold() on each token; original data are unchanged.\',\n    \'capitalization\': \'First character of the original first token is uppercase (str.isupper). Entity denominator: all mentions of that type. Sentence-O denominator: sentences whose token at index 0 has O; no punctuation skipping.\',\n    \'transition\': \'Only adjacent tokens within a sentence; no transitions across sentences. Probability divides by outgoing transitions for that label. Zero denominator is null (N/A).\',\n    \'ambiguity\': \'Exact token forms or complete entity surfaces with at least two distinct entity types WITHIN a split. O is excluded from entity types; O occurrence counts are supplemental.\',\n    \'vocabulary\': \'Includes punctuation and digits. Directed source-to-target type coverage = intersection / target vocabulary size; token coverage weights target token occurrences. Jaccard = intersection / union.\',\n    \'top_entities\': \'Top 10 per split and type, by mention count descending then token tuple ascending; plots show training entities.\',\n    \'paper_sentences\': \'Compare HF sentences directly to 14041/3250/3453; also compare HF sentences + document count to paper sentence count. DOCSTART explanation is an inference, not a reconstruction of the original files.\',\n    \'test_use\': \'Test statistics are descriptive only and must not select features, hyperparameters, preprocessing or models. No prediction scores are computed.\',\n}\n\n\ndef ratio(numerator, denominator):\n    return numerator / denominator if denominator else None\n\n\ndef extract_entities(tokens, tags):\n    """Return (start, end, type, token_tuple), rejecting illegal IOB2."""\n    if len(tokens) != len(tags):\n        raise ValueError(\'Token/tag length mismatch.\')\n    spans = []\n    start = kind = None\n    for index, tag in enumerate([*tags, \'O\']):\n        if tag != \'O\' and (tag[:2] not in (\'B-\', \'I-\') or tag[2:] not in TYPES):\n            raise ValueError(f\'Unknown tag: {tag}\')\n        if tag.startswith(\'I-\'):\n            if start is None or kind != tag[2:]:\n                raise ValueError(\'Illegal IOB2 continuation.\')\n        else:\n            if start is not None:\n                spans.append((start, index, kind, tuple(tokens[start:index])))\n                start = kind = None\n            if tag.startswith(\'B-\'):\n                start, kind = index, tag[2:]\n    return spans\n\n\ndef _normal_surface(surface, mode):\n    return surface if mode == \'exact\' else tuple(token.casefold() for token in surface)\n\n\ndef analyze(dataset, labels):\n    """One deterministic traversal; keeps distributions/aggregates rather than full rows."""\n    tables = {name: [] for name in (\'summary\', \'labels\', \'sentence_lengths\', \'entity_lengths\',\n              \'top_entities\', \'transitions\', \'capitalization\', \'unseen\', \'vocabulary\',\n              \'ambiguous_tokens\', \'ambiguous_entities\', \'order_spotchecks\', \'paper_comparison\')}\n    entities_by_split, vocab_by_split = {}, {}\n    for split in SPLITS:\n        rows = dataset[split]\n        label_counts, lengths, transitions, vocab = Counter(), Counter(), Counter(), Counter()\n        entity_counts = {kind: Counter() for kind in TYPES}\n        entity_lengths = {kind: Counter() for kind in TYPES}\n        caps = Counter()\n        token_types, surface_types = defaultdict(Counter), defaultdict(Counter)\n        outside_tokens = Counter()\n        docs, spots = set(), set(range(min(3, len(rows))))\n        middle = len(rows) // 2\n        spots.update(range(max(0, middle - 1), min(len(rows), middle + 2)))\n        previous_doc = None\n        boundary_found = False\n        for index, row in enumerate(rows):\n            tokens = row[\'tokens\']\n            tags = [labels[tag] for tag in row[\'ner_tags\']]\n            spans = extract_entities(tokens, tags)\n            label_counts.update(tags)\n            lengths[len(tokens)] += 1\n            vocab.update(tokens)\n            transitions.update(zip(tags, tags[1:]))\n            docs.add(row[\'document_id\'])\n            if previous_doc is not None and row[\'document_id\'] != previous_doc and not boundary_found:\n                spots.update([index - 1, index])\n                boundary_found = True\n            previous_doc = row[\'document_id\']\n            if tags and tags[0] == \'O\':\n                caps[\'sentence_O_total\'] += 1\n                caps[\'sentence_O_upper\'] += int(tokens[0][:1].isupper())\n            for token, tag in zip(tokens, tags):\n                if tag == \'O\':\n                    outside_tokens[token] += 1\n                else:\n                    token_types[token][tag[2:]] += 1\n            for start, end, kind, surface in spans:\n                entity_counts[kind][surface] += 1\n                entity_lengths[kind][end - start] += 1\n                surface_types[surface][kind] += 1\n                caps[kind] += int(surface[0][:1].isupper())\n            if len(spans) != sum(tag.startswith(\'B-\') for tag in tags):\n                raise ValueError(\'Span/B-tag count mismatch.\')\n        total_tokens = sum(vocab.values())\n        counts = {kind: sum(entity_counts[kind].values()) for kind in TYPES}\n        if sum(label_counts.values()) != total_tokens or sum(transitions.values()) != total_tokens - len(rows):\n            raise ValueError(\'Count/transition conservation failed.\')\n        summary = dict(split=split, sentences=len(rows), tokens=total_tokens, documents=len(docs),\n                       entities=sum(counts.values()), **counts)\n        tables[\'summary\'].append(summary)\n        for metric in (\'sentences\', \'tokens\', \'documents\', *TYPES):\n            expected = PAPER[split][metric]\n            tables[\'paper_comparison\'].append(dict(split=split, metric=metric,\n                observed=summary[metric], reference=expected, difference=summary[metric] - expected))\n        adjusted = len(rows) + len(docs)\n        tables[\'paper_comparison\'].append(dict(split=split, metric=\'sentences_plus_documents\',\n            observed=adjusted, reference=PAPER[split][\'paper_sentences\'],\n            difference=adjusted - PAPER[split][\'paper_sentences\']))\n        tables[\'paper_comparison\'].append(dict(split=split, metric=\'raw_sentences_vs_paper\',\n            observed=len(rows), reference=PAPER[split][\'paper_sentences\'],\n            difference=len(rows) - PAPER[split][\'paper_sentences\'], expected_difference=-PAPER[split][\'documents\']))\n        for label in labels:\n            tables[\'labels\'].append(dict(split=split, label=label, count=label_counts[label],\n                                        proportion=ratio(label_counts[label], total_tokens)))\n            outgoing = sum(transitions[(label, other)] for other in labels)\n            for other in labels:\n                n = transitions[(label, other)]\n                tables[\'transitions\'].append(dict(split=split, previous=label, following=other,\n                                                  count=n, probability=ratio(n, outgoing)))\n        for length, count in sorted(lengths.items()):\n            tables[\'sentence_lengths\'].append(dict(split=split, length=length, count=count))\n        for kind in TYPES:\n            for length, count in sorted(entity_lengths[kind].items()):\n                tables[\'entity_lengths\'].append(dict(split=split, type=kind, length=length,\n                                                     count=count, proportion=ratio(count, counts[kind])))\n            for rank, (surface, count) in enumerate(sorted(entity_counts[kind].items(),\n                                                           key=lambda x: (-x[1], x[0]))[:10], 1):\n                tables[\'top_entities\'].append(dict(split=split, type=kind, rank=rank,\n                                                  surface=\' \'.join(surface), count=count))\n            tables[\'capitalization\'].append(dict(split=split, metric=kind, uppercase=caps[kind],\n                                                 total=counts[kind], proportion=ratio(caps[kind], counts[kind])))\n        tables[\'capitalization\'].append(dict(split=split, metric=\'sentence_initial_O\',\n            uppercase=caps[\'sentence_O_upper\'], total=caps[\'sentence_O_total\'],\n            proportion=ratio(caps[\'sentence_O_upper\'], caps[\'sentence_O_total\'])))\n        for table, forms in [(\'ambiguous_tokens\', token_types), (\'ambiguous_entities\', surface_types)]:\n            ambiguous = [(form, values) for form, values in forms.items() if len(values) > 1]\n            for form, values in sorted(ambiguous, key=lambda x: (-sum(x[1].values()), x[0])):\n                item = dict(split=split, surface=form if table == \'ambiguous_tokens\' else \' \'.join(form),\n                            types=\'|\'.join(sorted(values)), total=sum(values.values()),\n                            **{kind: values[kind] for kind in TYPES})\n                if table == \'ambiguous_tokens\':\n                    item[\'O_count\'] = outside_tokens[form]\n                tables[table].append(item)\n        for index in sorted(spots):\n            row = rows[index]\n            tables[\'order_spotchecks\'].append(dict(split=split, row=index, id=row[\'id\'],\n                document_id=row[\'document_id\'], sentence_id=row[\'sentence_id\'],\n                preview=\' \'.join(row[\'tokens\'][:24])))\n        entities_by_split[split], vocab_by_split[split] = entity_counts, vocab\n\n    for mode in (\'exact\', \'casefold\'):\n        train_by_type = {kind: {_normal_surface(s, mode) for s in entities_by_split[\'train\'][kind]}\n                         for kind in TYPES}\n        train_any = set().union(*train_by_type.values())\n        for split in (\'validation\', \'test\'):\n            for kind in TYPES:\n                target = Counter()\n                for surface, count in entities_by_split[split][kind].items():\n                    target[_normal_surface(surface, mode)] += count\n                unseen = {s for s in target if s not in train_any}\n                unseen_typed = {s for s in target if s not in train_by_type[kind]}\n                n, unique = sum(target.values()), len(target)\n                tables[\'unseen\'].append(dict(split=split, normalization=mode, type=kind,\n                    mentions=n, unique_surfaces=unique,\n                    unseen_mentions=sum(target[s] for s in unseen), unseen_unique=len(unseen),\n                    mention_rate=ratio(sum(target[s] for s in unseen), n), unique_rate=ratio(len(unseen), unique),\n                    unseen_typed_mentions=sum(target[s] for s in unseen_typed), unseen_typed_unique=len(unseen_typed),\n                    typed_mention_rate=ratio(sum(target[s] for s in unseen_typed), n),\n                    typed_unique_rate=ratio(len(unseen_typed), unique)))\n        vocabs = {}\n        for split in SPLITS:\n            values = Counter()\n            for token, count in vocab_by_split[split].items():\n                values[token if mode == \'exact\' else token.casefold()] += count\n            vocabs[split] = values\n        for source in SPLITS:\n            for target in SPLITS:\n                a, b = set(vocabs[source]), set(vocabs[target])\n                overlap = a & b\n                covered = sum(vocabs[target][t] for t in overlap)\n                tables[\'vocabulary\'].append(dict(normalization=mode, source=source, target=target,\n                    source_types=len(a), target_types=len(b), shared_types=len(overlap), union_types=len(a | b),\n                    type_coverage=ratio(len(overlap), len(b)), jaccard=ratio(len(overlap), len(a | b)),\n                    covered_target_tokens=covered, target_tokens=sum(vocabs[target].values()),\n                    token_coverage=ratio(covered, sum(vocabs[target].values()))))\n    for row in tables[\'paper_comparison\']:\n        row.setdefault(\'expected_difference\', 0)\n        row[\'matches_expected\'] = row[\'difference\'] == row[\'expected_difference\']\n    return {\'definitions\': DEFINITIONS, \'labels_from_metadata\': list(labels), \'tables\': tables,\n            \'paper_comparison_passed\': all(row[\'matches_expected\'] for row in tables[\'paper_comparison\'])}\n\n\ndef ensure_dataset(project_root, accepted, run_id):\n    """Reuse the accepted Step 1 files, or redownload exactly the same pinned content."""\n    import step1_data as data_tools\n    from datasets import load_from_disk\n\n    project_root = Path(project_root)\n    path = project_root / \'data\' / Path(accepted[\'saved_data\'][\'path\']).name\n    if path.is_dir():\n        for record in accepted[\'saved_data\'][\'files\']:\n            file = path / record[\'path\']\n            if not file.is_file() or data_tools.file_sha256(file) != record[\'sha256\']:\n                raise ValueError(\'Saved Step 1 data changed. Stop and report this error.\')\n        dataset = load_from_disk(str(path))\n        source = accepted[\'source\']\n        recovery = \'reused-verified-step1-files\'\n    else:\n        dataset, source = data_tools.load_pinned_dataset(project_root)\n        if source[\'source_files\'] != accepted[\'source\'][\'source_files\']:\n            raise ValueError(\'Pinned downloads differ from the accepted Step 1 source hashes.\')\n        recovery = \'redownloaded-pinned-step1-source\'\n    audit = data_tools.audit_dataset(dataset)\n    if audit != accepted[\'audit\']:\n        raise ValueError(\'Dataset differs from the accepted Step 1 ordered audit.\')\n    if recovery.startswith(\'redownloaded\'):\n        saved = data_tools.save_and_verify(dataset, project_root, run_id, audit)\n    else:\n        saved = {**accepted[\'saved_data\'], \'path\': str(path)}\n    record = dict(source=source, saved_data=saved, audit=audit, recovery=recovery, run_id=run_id)\n    data_tools.write_json(project_root / \'config\' / \'dataset_latest.json\', record)\n    return dataset, record\n\n\ndef write_tables(report, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=False)\n    for name, rows in report[\'tables\'].items():\n        path = destination / f\'{name}.csv\'\n        with path.open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as stream:\n            fields = list(rows[0]) if rows else [\'split\', \'surface\', \'types\', \'total\', *TYPES]\n            writer = csv.DictWriter(stream, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    (destination / \'eda_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    lines = [\'# Step 2 descriptive EDA\', \'\', \'No model training or prediction scoring.\', \'\',\n             \'Reference: https://aclanthology.org/W03-0419.pdf (Tables 1 and 2).\', \'\', \'## Definitions\', \'\']\n    lines.extend(f\'- {key}: {value}\' for key, value in report[\'definitions\'].items())\n    lines += [\'\', \'## Observed counts\', \'\', \'| Split | Sentences | Tokens | Entities | PER | ORG | LOC | MISC |\',\n              \'|---|---:|---:|---:|---:|---:|---:|---:|\']\n    for row in report[\'tables\'][\'summary\']:\n        lines.append(\'| \' + \' | \'.join(str(row[k]) for k in (\'split\', \'sentences\', \'tokens\', \'entities\', *TYPES)) + \' |\')\n    lines += [\'\', f"Paper count comparison passed: {report[\'paper_comparison_passed\']}.",\n              \'Raw paper sentence counts include an additional document-sized difference; see paper_comparison.csv.\',\n              \'Use train-only observations when considering features. Test statistics are descriptive, not a selection criterion.\',\n              \'PNG and vector PDF figures are stored in the corresponding figures/step2_<run_id> directory.\']\n    (destination / \'EDA_README.md\').write_text(\'\\n\'.join(lines) + \'\\n\', encoding=\'utf-8\')\n\n\nFIGURE_NAMES = (\'01_label_distribution\', \'02_sentence_lengths\', \'03_entity_lengths\',\n                \'04_top_entities_train\', \'05_tag_transitions\', \'06_capitalization\',\n                \'07_unseen_entities\', \'08_vocabulary_overlap\', \'09_ambiguity_train\')\n\n\ndef plot_report(report, destination):\n    """Nine report figures, each exported as a PNG and a vector PDF."""\n    import matplotlib.pyplot as plt\n    import numpy as np\n    from matplotlib.ticker import PercentFormatter, MaxNLocator\n    from textwrap import fill\n\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    tables, labels = report[\'tables\'], report[\'labels_from_metadata\']\n    colors = [\'#2670AA\', \'#D17B22\', \'#36846B\', \'#A75085\']\n    paths = []\n    style = {\'font.family\': \'DejaVu Sans\', \'font.size\': 10, \'axes.spines.top\': False,\n             \'axes.spines.right\': False, \'savefig.facecolor\': \'white\', \'figure.facecolor\': \'white\'}\n\n    def selected(name, **where):\n        return [r for r in tables[name] if all(r[k] == v for k, v in where.items())]\n\n    def save(fig, name):\n        fig.savefig(destination / (name + \'.png\'), dpi=170, bbox_inches=\'tight\')\n        fig.savefig(destination / (name + \'.pdf\'), bbox_inches=\'tight\')\n        paths.append(str(destination / (name + \'.png\')))\n        plt.close(fig)\n\n    def number(value):\n        return float(\'nan\') if value is None else value\n\n    with plt.rc_context(style):\n        fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), layout=\'constrained\', sharey=True)\n        for ax, split, color in zip(axes, SPLITS, colors):\n            rows = selected(\'labels\', split=split)\n            ax.bar([r[\'label\'] for r in rows], [r[\'proportion\'] for r in rows], color=color)\n            ax.set(title=split, xlabel=\'Token label (includes O)\')\n            ax.tick_params(axis=\'x\', rotation=55)\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n        axes[0].set_ylabel(\'Share of tokens\')\n        fig.suptitle(\'NER label distribution\')\n        save(fig, FIGURE_NAMES[0])\n\n        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout=\'constrained\')\n        maximum = max(r[\'length\'] for r in tables[\'sentence_lengths\'])\n        bins = np.arange(0.5, maximum + 5.5, 5)\n        for ax, split, color in zip(axes, SPLITS, colors):\n            rows = selected(\'sentence_lengths\', split=split)\n            ax.hist([r[\'length\'] for r in rows], bins=bins, weights=[r[\'count\'] for r in rows], color=color)\n            ax.set(title=f\'{split} (max {max(r["length"] for r in rows)})\', xlabel=\'Tokens per sentence\', ylabel=\'Sentences\', xlim=(0, maximum + 1))\n        fig.suptitle(\'Sentence lengths — all sentences, no tail truncation\')\n        save(fig, FIGURE_NAMES[1])\n\n        fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout=\'constrained\')\n        for ax, kind in zip(axes.flat, TYPES):\n            kind_rows = selected(\'entity_lengths\', type=kind)\n            maximum = max((r[\'length\'] for r in kind_rows), default=1)\n            x = np.arange(1, maximum + 1)\n            for split, color in zip(SPLITS, colors):\n                rows = selected(\'entity_lengths\', split=split, type=kind)\n                counts = {r[\'length\']: r[\'count\'] for r in rows}\n                total = sum(counts.values())\n                probabilities = [counts.get(length, 0) / total if total else np.nan for length in x]\n                ax.plot(x, probabilities, marker=\'o\', ms=3, label=split, color=color)\n            ax.set(title=kind, xlabel=\'Tokens per entity\', ylabel=\'Share within split/type\',\n                   ylim=(0, 1.05), xlim=(.5, maximum + .5))\n            ax.xaxis.set_major_locator(MaxNLocator(integer=True))\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n            ax.legend(fontsize=8)\n        fig.suptitle(\'Entity length distributions\')\n        save(fig, FIGURE_NAMES[2])\n\n        fig, axes = plt.subplots(2, 2, figsize=(16, 12), layout=\'constrained\')\n        for ax, kind, color in zip(axes.flat, TYPES, colors):\n            rows = selected(\'top_entities\', split=\'train\', type=kind)\n            ax.barh(range(len(rows)), [r[\'count\'] for r in rows], color=color)\n            ax.set_yticks(range(len(rows)), [fill(r[\'surface\'], 26) for r in rows], fontsize=9)\n            ax.invert_yaxis()\n            ax.set(title=kind, xlabel=\'Entity mentions in training data\')\n        fig.suptitle(\'Ten most frequent entity surfaces per type (train only)\')\n        save(fig, FIGURE_NAMES[3])\n\n        fig, axes = plt.subplots(1, 3, figsize=(20, 7), layout=\'constrained\')\n        for ax, split in zip(axes, SPLITS):\n            lookup = {(r[\'previous\'], r[\'following\']): r[\'probability\'] for r in selected(\'transitions\', split=split)}\n            matrix = np.array([[number(lookup[(a, b)]) for b in labels] for a in labels])\n            im = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=1, cmap=\'Blues\')\n            ax.set_xticks(range(len(labels)), labels, rotation=60, ha=\'right\')\n            ax.set_yticks(range(len(labels)), labels)\n            ax.set(title=split, xlabel=\'Next label\', ylabel=\'Current label\')\n            for i in range(len(labels)):\n                for j in range(len(labels)):\n                    v = matrix[i, j]\n                    ax.text(j, i, \'N/A\' if np.isnan(v) else f\'{100*v:.0f}\', ha=\'center\', va=\'center\',\n                            fontsize=7, color=\'white\' if v > .55 else \'#172B40\')\n        fig.colorbar(im, ax=axes, shrink=.65, label=\'Row probability\')\n        fig.suptitle(\'Within-sentence label transitions — cell values in percent\')\n        save(fig, FIGURE_NAMES[4])\n\n        fig, ax = plt.subplots(figsize=(12, 5.5), layout=\'constrained\')\n        categories = [*TYPES, \'sentence_initial_O\']\n        x = np.arange(len(categories))\n        for i, (split, color) in enumerate(zip(SPLITS, colors)):\n            lookup = {r[\'metric\']: r for r in selected(\'capitalization\', split=split)}\n            ax.bar(x + (i-1)*.25, [number(lookup[k][\'proportion\']) for k in categories], width=.24, color=color, label=split)\n        ax.set_xticks(x, [*TYPES, \'Sentence-first O token\'])\n        ax.set(ylabel=\'Uppercase first character / eligible occurrences\', ylim=(0, 1.05),\n               title=\'Capitalization of entity starts and sentence-initial non-entities\')\n        ax.yaxis.set_major_formatter(PercentFormatter(1))\n        ax.legend()\n        save(fig, FIGURE_NAMES[5])\n\n        fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout=\'constrained\', sharey=True)\n        x = np.arange(len(TYPES))\n        for ax, rate, title in zip(axes, (\'mention_rate\', \'unique_rate\'), (\'By mention occurrence\', \'By unique surface\')):\n            for i, (split, color) in enumerate(zip((\'validation\', \'test\'), colors[1:])):\n                lookup = {r[\'type\']: r for r in selected(\'unseen\', split=split, normalization=\'exact\')}\n                ax.bar(x + (i-.5)*.34, [number(lookup[k][rate]) for k in TYPES], width=.33, label=split, color=color)\n            ax.set_xticks(x, TYPES)\n            ax.set(title=title, ylabel=\'Unseen proportion\', ylim=(0, 1.05))\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n            ax.legend()\n        fig.suptitle(\'Entity surfaces unseen among training entities (exact case, any type)\')\n        save(fig, FIGURE_NAMES[6])\n\n        fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), layout=\'constrained\')\n        for ax, mode in zip(axes, (\'exact\', \'casefold\')):\n            lookup = {(r[\'source\'], r[\'target\']): r[\'type_coverage\'] for r in selected(\'vocabulary\', normalization=mode)}\n            matrix = np.array([[number(lookup[(a, b)]) for b in SPLITS] for a in SPLITS])\n            im = ax.imshow(matrix, cmap=\'YlGnBu\', vmin=0, vmax=1)\n            ax.set_xticks(range(3), SPLITS)\n            ax.set_yticks(range(3), SPLITS)\n            ax.set(title=mode, xlabel=\'Target vocabulary (denominator)\', ylabel=\'Source vocabulary\')\n            for i in range(3):\n                for j in range(3):\n                    ax.text(j, i, f\'{matrix[i,j]:.1%}\', ha=\'center\', va=\'center\', color=\'white\' if matrix[i,j] > .65 else \'black\')\n        fig.colorbar(im, ax=axes, shrink=.75, label=\'Target type coverage\')\n        fig.suptitle(\'Directed vocabulary overlap (includes punctuation and digits)\')\n        save(fig, FIGURE_NAMES[7])\n\n        fig, axes = plt.subplots(1, 2, figsize=(16, 10), layout=\'constrained\')\n        for ax, name, title in zip(axes, (\'ambiguous_tokens\', \'ambiguous_entities\'), (\'Token forms\', \'Complete entity surfaces\')):\n            rows = selected(name, split=\'train\')[:15]\n            if not rows:\n                ax.text(.5, .5, \'No multi-type forms\', ha=\'center\', va=\'center\', transform=ax.transAxes)\n                ax.set_title(title)\n                continue\n            left = np.zeros(len(rows))\n            for kind, color in zip(TYPES, colors):\n                values = np.array([r[kind] for r in rows])\n                ax.barh(range(len(rows)), values, left=left, label=kind, color=color)\n                left += values\n            ax.set_yticks(range(len(rows)), [fill(r[\'surface\'], 25) for r in rows], fontsize=9)\n            ax.invert_yaxis()\n            ax.set(title=title, xlabel=\'Entity-token occurrences\' if name == \'ambiguous_tokens\' else \'Entity mentions\')\n            ax.legend(fontsize=8)\n        fig.suptitle(\'Multi-type ambiguity in training data (up to 15 most frequent forms; O excluded)\')\n        save(fig, FIGURE_NAMES[8])\n    return paths\n', 'step3_alignment.py': '"""Lossless word/subword alignment, whole-word chunks and prediction reconstruction."""\nimport hashlib\nimport json\nfrom collections import Counter\nfrom numbers import Integral\nfrom pathlib import Path\n\nIGNORE_INDEX = -100\nMAX_LENGTH = 256  # Includes special tokens; fixed before observing tokenized test lengths.\nTOKENIZER_SPECS = {\n    \'bert_base_cased\': {\n        \'repo_id\': \'google-bert/bert-base-cased\',\n        \'revision\': \'cd5ef92a9fb2f889e972770a36d4ed042daf221e\',\n        \'files\': [\'config.json\', \'tokenizer_config.json\', \'tokenizer.json\', \'vocab.txt\'],\n        \'kwargs\': {},\n    },\n    \'roberta_base\': {\n        \'repo_id\': \'FacebookAI/roberta-base\',\n        \'revision\': \'e2da8e2f811d1448a5b465c236feacd80ffbac7b\',\n        \'files\': [\'config.json\', \'tokenizer_config.json\', \'tokenizer.json\', \'vocab.json\', \'merges.txt\'],\n        \'kwargs\': {\'add_prefix_space\': True},\n    },\n}\nPOLICY = {\n    \'max_length\': MAX_LENGTH,\n    \'includes_special_tokens\': True,\n    \'scheme\': \'IOB2 labels retained from metadata; no relabeling at chunk boundaries.\',\n    \'supervision\': \'First subword of each original word only; continuation/special/padding labels are -100.\',\n    \'chunking\': \'Greedy nonoverlapping complete-word chunks based only on tokenized word lengths; never gold entities.\',\n    \'oversized_word\': \'Raise an error if one word exceeds the content budget or produces no subword; never silently discard it.\',\n    \'evaluation\': \'Reassemble each original sentence in word order before entity metrics; do not score chunks separately.\',\n    \'context_limit\': \'Whole-word chunking can cut through an entity and limits cross-chunk context; original gold labels are preserved.\',\n    \'padding\': \'Dynamic right-padding per batch; strip alignment metadata before passing a batch to a model.\',\n    \'model_work\': \'Tokenizer preparation only: no model weights, training, predictions or F1 scoring.\',\n}\n\n\ndef load_tokenizer(key, project_root):\n    from huggingface_hub import snapshot_download\n    from transformers import AutoTokenizer\n    from step1_data import file_sha256\n\n    spec = TOKENIZER_SPECS[key]\n    snapshot = Path(snapshot_download(repo_id=spec[\'repo_id\'], revision=spec[\'revision\'],\n        allow_patterns=spec[\'files\'], cache_dir=str(Path(project_root) / \'data\' / \'tokenizer_cache\'), token=False))\n    files = []\n    for filename in spec[\'files\']:\n        path = snapshot / filename\n        if not path.is_file():\n            raise FileNotFoundError(f\'Missing pinned tokenizer file: {filename}\')\n        files.append(dict(path=filename, size_bytes=path.stat().st_size, sha256=file_sha256(path)))\n    tokenizer = AutoTokenizer.from_pretrained(str(snapshot), use_fast=True, local_files_only=True,\n                                               trust_remote_code=False, **spec[\'kwargs\'])\n    tokenizer.padding_side = \'right\'\n    if not tokenizer.is_fast or tokenizer.pad_token_id is None:\n        raise ValueError(\'A fast tokenizer with a padding token is required.\')\n    if MAX_LENGTH > tokenizer.model_max_length:\n        raise ValueError(\'Chosen maximum exceeds the tokenizer model limit.\')\n    record = dict(key=key, **spec, source_files=files, tokenizer_class=type(tokenizer).__name__,\n                  model_max_length=tokenizer.model_max_length, padding_side=tokenizer.padding_side,\n                  special_tokens_per_sentence=tokenizer.num_special_tokens_to_add(pair=False))\n    return tokenizer, record\n\n\ndef whole_word_ranges(piece_counts, budget):\n    """Return contiguous [start,end) word ranges, with no overlap or truncation."""\n    if budget < 1 or not piece_counts:\n        raise ValueError(\'Positive content budget and at least one word are required.\')\n    ranges, start, used = [], 0, 0\n    for index, count in enumerate(piece_counts):\n        if count < 1 or count > budget:\n            raise ValueError(f\'Word {index} has {count} pieces, outside content budget 1..{budget}.\')\n        if used + count > budget:\n            ranges.append((start, index))\n            start, used = index, 0\n        used += count\n    ranges.append((start, len(piece_counts)))\n    return ranges\n\n\ndef encode_sentence(row, sentence_index, tokenizer, label_names, max_length=MAX_LENGTH):\n    words, gold = row[\'tokens\'], row[\'ner_tags\']\n    if not words or len(words) != len(gold):\n        raise ValueError(\'Missing words or token/gold length mismatch.\')\n    if any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < len(label_names) for v in gold):\n        raise ValueError(\'Invalid gold label index.\')\n    special_count = tokenizer.num_special_tokens_to_add(pair=False)\n    if max_length > tokenizer.model_max_length:\n        raise ValueError(\'Chosen max_length exceeds the model limit.\')\n    flat = tokenizer(words, is_split_into_words=True, add_special_tokens=False, truncation=False,\n                     padding=False, return_attention_mask=False, return_token_type_ids=False, verbose=False)\n    flat_ids, word_ids = flat[\'input_ids\'], flat.word_ids()\n    if len(flat_ids) != len(word_ids) or any(w is None for w in word_ids):\n        raise ValueError(\'Unexpected special/unmapped token in plain word encoding.\')\n    runs = [w for i, w in enumerate(word_ids) if i == 0 or word_ids[i-1] != w]\n    if runs != list(range(len(words))):\n        raise ValueError(\'Tokenizer omitted, reordered or fragmented a word mapping.\')\n    counts = Counter(word_ids)\n    pieces = [counts[i] for i in range(len(words))]\n    boundaries = [0]\n    for count in pieces:\n        boundaries.append(boundaries[-1] + count)\n    ranges = whole_word_ranges(pieces, max_length - special_count)\n    chunks = []\n    for chunk_index, (start, end) in enumerate(ranges):\n        encoded = tokenizer(words[start:end], is_split_into_words=True, add_special_tokens=True,\n                            truncation=False, padding=False, return_special_tokens_mask=True)\n        local_ids = encoded.word_ids()\n        if [w for i,w in enumerate(local_ids) if w is not None and (i == 0 or local_ids[i-1] != w)] != list(range(end-start)):\n            raise ValueError(\'Chunk word mapping changed.\')\n        content = [token for token,w in zip(encoded[\'input_ids\'], local_ids) if w is not None]\n        if content != flat_ids[boundaries[start]:boundaries[end]]:\n            raise ValueError(\'Chunk tokenization differs from full-sentence tokenization.\')\n        if len(encoded[\'input_ids\']) > max_length:\n            raise ValueError(\'Whole-word chunk still exceeds max_length.\')\n        labels, mapped, previous = [], [], None\n        for w, special in zip(local_ids, encoded[\'special_tokens_mask\']):\n            if (w is None) != bool(special):\n                raise ValueError(\'Special-token mask disagrees with word mapping.\')\n            mapped.append(-1 if w is None else start + w)\n            labels.append(int(gold[start+w]) if w is not None and w != previous else IGNORE_INDEX)\n            previous = w\n        chunk = {k:list(encoded[k]) for k in tokenizer.model_input_names if k in encoded}\n        chunk.update(labels=labels, word_ids=mapped, sentence_index=sentence_index,\n                     sentence_id=str(row[\'id\']), chunk_index=chunk_index, word_start=start, word_end=end)\n        chunks.append(chunk)\n    stats = dict(sentence_index=sentence_index, sentence_id=str(row[\'id\']), words=len(words),\n                 full_length=len(flat_ids)+special_count, chunks=len(chunks),\n                 multi_piece_words=sum(n > 1 for n in pieces),\n                 unknown_pieces=sum(v == tokenizer.unk_token_id for v in flat_ids))\n    return chunks, stats\n\n\ndef reassemble_word_predictions(chunks, prediction_ids, word_counts, num_labels):\n    """Class ids -> original words. Ignores special/continuation/padding predictions.\n\n    Alignment uses word_ids, not gold labels. Prediction rows must follow chunk rows.\n    This also validates a gold-label round trip without computing a model metric.\n    """\n    if len(chunks) != len(prediction_ids):\n        raise ValueError(\'Prediction/chunk count mismatch.\')\n    restored = [[None] * n for n in word_counts]\n    for chunk, predictions in zip(chunks, prediction_ids):\n        if len(predictions) < len(chunk[\'input_ids\']):\n            raise ValueError(\'Prediction row shorter than its encoded chunk.\')\n        sentence = chunk[\'sentence_index\']\n        if not 0 <= sentence < len(restored):\n            raise ValueError(\'Unknown sentence index.\')\n        previous = -1\n        selected = []\n        for index, word in enumerate(chunk[\'word_ids\']):\n            if word >= 0 and word != previous:\n                if not 0 <= word < len(restored[sentence]):\n                    raise ValueError(\'Word index outside original sentence.\')\n                value = predictions[index]\n                if isinstance(value, bool) or not isinstance(value, Integral) or not 0 <= value < num_labels:\n                    raise ValueError(\'Invalid prediction at a first-subword position.\')\n                if restored[sentence][word] is not None:\n                    raise ValueError(\'Duplicate prediction for an original word.\')\n                restored[sentence][word] = int(value)\n                selected.append(word)\n            previous = word\n        if selected != list(range(chunk[\'word_start\'], chunk[\'word_end\'])):\n            raise ValueError(\'Chunk word range differs from first-subword mapping.\')\n    if any(value is None for row in restored for value in row):\n        raise ValueError(\'At least one original word has no prediction.\')\n    return restored\n\n\ndef model_features(chunk, tokenizer):\n    """Keep metadata out of the model/collator input, retaining it in the Arrow sidecar."""\n    return {k:chunk[k] for k in [*tokenizer.model_input_names, \'labels\'] if k in chunk}\n\n\ndef validate_chunks(chunks, source_rows, num_labels, max_length=MAX_LENGTH):\n    counts = [len(row[\'tokens\']) for row in source_rows]\n    labels = [row[\'ner_tags\'] for row in source_rows]\n    special = continuation = supervised = 0\n    lengths = []\n    for chunk in chunks:\n        n = len(chunk[\'input_ids\'])\n        if n > max_length or n < 1 or any(len(chunk[k]) != n for k in (\'attention_mask\',\'labels\',\'word_ids\')):\n            raise ValueError(\'Chunk lengths are inconsistent.\')\n        if \'token_type_ids\' in chunk and len(chunk[\'token_type_ids\']) != n:\n            raise ValueError(\'Token-type ids have an inconsistent length.\')\n        if any(isinstance(w, bool) or not isinstance(w, Integral) or w < -1 for w in chunk[\'word_ids\']):\n            raise ValueError(\'Invalid stored word index.\')\n        if any(v != 1 for v in chunk[\'attention_mask\']):\n            raise ValueError(\'Saved chunks must be unpadded.\')\n        previous = -1\n        for word, value in zip(chunk[\'word_ids\'], chunk[\'labels\']):\n            first = word >= 0 and word != previous\n            if not first:\n                if value != IGNORE_INDEX:\n                    raise ValueError(\'Special or continuation token carries supervision.\')\n                special += int(word < 0)\n                continuation += int(word >= 0)\n            else:\n                supervised += 1\n            previous = word\n        lengths.append(n)\n    restored = reassemble_word_predictions(chunks, [r[\'labels\'] for r in chunks], counts, num_labels)\n    if restored != labels or supervised != sum(counts):\n        raise ValueError(\'Gold labels did not round-trip for every original word exactly once.\')\n    return dict(original_words=sum(counts), supervised_words=supervised, dropped_words=0,\n                duplicated_words=0, masked_special_tokens=special, masked_continuation_tokens=continuation,\n                max_chunk_length=max(lengths, default=0), gold_roundtrip_passed=True)\n\n\ndef padding_check(tokenizer, chunks):\n    from transformers import DataCollatorForTokenClassification\n    import numpy as np\n\n    chosen = [min(chunks, key=lambda x:len(x[\'input_ids\'])), max(chunks, key=lambda x:len(x[\'input_ids\']))]\n    collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True,\n                                                  label_pad_token_id=IGNORE_INDEX, return_tensors=\'np\')\n    batch = collator([model_features(row, tokenizer) for row in chosen])\n    for i, row in enumerate(chosen):\n        n = len(row[\'input_ids\'])\n        if list(batch[\'labels\'][i,:n]) != row[\'labels\']:\n            raise ValueError(\'Collator changed original labels.\')\n    masked = batch[\'attention_mask\'] == 0\n    if not np.all(batch[\'labels\'][masked] == IGNORE_INDEX):\n        raise ValueError(\'Padding labels must be -100.\')\n    return dict(batch_shape=list(batch[\'input_ids\'].shape), padding_positions=int(masked.sum()), padding_labels_verified=True)\n\n\ndef records_sha256(records):\n    digest = hashlib.sha256()\n    for row in records:\n        digest.update((json.dumps(row,sort_keys=True,separators=(\',\',\':\'))+\'\\n\').encode(\'utf-8\'))\n    return digest.hexdigest()\n\n\ndef synthetic_alignment_check(tokenizer, label_names):\n    """Exercise multi-piece words, forced long-entity splitting and dynamic padding."""\n    ids = {label: i for i, label in enumerate(label_names)}\n    rows = [\n        dict(id=\'synthetic-short\', tokens=[\'Jane\', \'Villanueva\', \'visited\', \'Hong\', \'Kong\', \'.\'],\n             ner_tags=[ids[t] for t in [\'B-PER\', \'I-PER\', \'O\', \'B-LOC\', \'I-LOC\', \'O\']]),\n        dict(id=\'synthetic-long\', tokens=[\'Jane\'] + [\'Villanueva\']*180 + [\'.\'],\n             ner_tags=[ids[\'B-PER\']] + [ids[\'I-PER\']]*180 + [ids[\'O\']]),\n        dict(id=\'synthetic-symbols\', tokens=[\'U.N.\', \'München\', \'中文\', \'[CLS]\', \'<s>\', \'1996-08-22\'],\n             ner_tags=[ids[\'O\']]*6),\n    ]\n    chunks, lengths = [], []\n    for index, row in enumerate(rows):\n        encoded, info = encode_sentence(row, index, tokenizer, label_names, max_length=64)\n        chunks.extend(encoded); lengths.append(info)\n    audit = validate_chunks(chunks, rows, len(label_names), max_length=64)\n    padded = padding_check(tokenizer, chunks)\n    if lengths[1][\'chunks\'] <= 1 or audit[\'masked_continuation_tokens\'] == 0 or padded[\'padding_positions\'] == 0:\n        raise ValueError(\'Synthetic test did not exercise splitting, continuation and padding.\')\n    restored = reassemble_word_predictions(chunks, [r[\'labels\'] + [0]*3 for r in chunks],\n                                          [len(r[\'tokens\']) for r in rows], len(label_names))\n    if restored != [r[\'ner_tags\'] for r in rows]:\n        raise ValueError(\'Synthetic reconstruction with padded predictions failed.\')\n    return dict(synthetic_only=True, forced_max_length=64, sentences=lengths, audit=audit, padding=padded)\n\n\ndef prepare_tokenizer_dataset(dataset, tokenizer, label_names, destination, key):\n    from datasets import Dataset, DatasetDict, load_from_disk\n    from step1_data import file_sha256\n\n    destination = Path(destination)\n    if destination.exists():\n        raise FileExistsError(\'Output exists. Reinitialize Step 3 to create a fresh run_id.\')\n    prepared, summaries, histograms, overflow, examples = {}, [], [], [], []\n    expected_hashes = {}\n    for split, source in dataset.items():\n        records, lengths = [], Counter()\n        multi = unknown = long_count = 0\n        for index, row in enumerate(source):\n            chunks, info = encode_sentence(row, index, tokenizer, label_names)\n            records.extend(chunks)\n            lengths[info[\'full_length\']] += 1\n            multi += info[\'multi_piece_words\']; unknown += info[\'unknown_pieces\']\n            if info[\'full_length\'] > MAX_LENGTH:\n                long_count += 1\n                overflow.append(dict(tokenizer=key, split=split, **info))\n            if split == \'train\' and (index < 2 or (info[\'chunks\'] > 1 and not any(e[\'example_kind\']==\'long_train\' for e in examples))):\n                for chunk in chunks:\n                    for pos, (piece, word, label) in enumerate(zip(tokenizer.convert_ids_to_tokens(chunk[\'input_ids\']),chunk[\'word_ids\'],chunk[\'labels\'])):\n                        examples.append(dict(tokenizer=key, example_kind=\'first_train\' if index<2 else \'long_train\',\n                            sentence_index=index, chunk_index=chunk[\'chunk_index\'], position=pos, subword=piece,\n                            word_index=word, original_word=\'\' if word<0 else row[\'tokens\'][word],\n                            label_id=label, label=\'IGNORED\' if label==IGNORE_INDEX else label_names[label]))\n            if (index+1) % 5000 == 0:\n                print(f\'{key}/{split}: aligned {index+1}/{len(source)} sentences\')\n        validation = validate_chunks(records, source, len(label_names))\n        padded = padding_check(tokenizer, records)\n        expected_hashes[split] = records_sha256(records)\n        prepared[split] = Dataset.from_list(records)\n        summaries.append(dict(tokenizer=key,split=split,sentences=len(source),chunks=len(records),\n            over_max_length_sentences=long_count,max_full_length=max(lengths),\n            multi_piece_words=multi,unknown_pieces=unknown,**validation,**padded,\n            ordered_alignment_sha256=expected_hashes[split]))\n        histograms.extend(dict(tokenizer=key,split=split,full_length=n,sentences=count) for n,count in sorted(lengths.items()))\n    result = DatasetDict(prepared)\n    result.save_to_disk(str(destination / \'dataset\'))\n    tokenizer.save_pretrained(str(destination / \'tokenizer\'))\n    reloaded = load_from_disk(str(destination / \'dataset\'))\n    for split, source in dataset.items():\n        if records_sha256(reloaded[split]) != expected_hashes[split]:\n            raise ValueError(\'Alignment files changed after save/read-back.\')\n        validate_chunks(reloaded[split], source, len(label_names))\n    files = [dict(path=p.relative_to(destination).as_posix(),size_bytes=p.stat().st_size,sha256=file_sha256(p))\n             for p in sorted(destination.rglob(\'*\')) if p.is_file()]\n    return dict(key=key, path=str(destination), summaries=summaries, length_histograms=histograms,\n                overflow=overflow, alignment_examples=examples, saved_files=files, read_back_verified=True)\n'}
ACCEPTED_DATA = {'source': {'requested_source': 'eriktks/conll2003', 'actual_source': 'tomaarsen/conll2003', 'revision': '3a0c0677d3177b6122fceed3239dda7a5de6a514', 'format': 'parquet', 'source_url': 'https://huggingface.co/datasets/tomaarsen/conll2003/tree/3a0c0677d3177b6122fceed3239dda7a5de6a514', 'reason': 'Development document permits Parquet fallback; datasets 4.0 removes loading scripts.', 'license_as_stated_by_card': 'other', 'source_files': [{'path': 'data/train-00000-of-00001.parquet', 'size_bytes': 1241085, 'sha256': '28924b53e43ec4d741cbef16d8a6245a77f404a77f18d44e256f95fa29acb59b'}, {'path': 'data/validation-00000-of-00001.parquet', 'size_bytes': 315597, 'sha256': '0aa0bd59cf9ac8f60252d8194a8a223b650daba50f8e5a79284abf85420bfe84'}, {'path': 'data/test-00000-of-00001.parquet', 'size_bytes': 287552, 'sha256': '9242163e112f99b43c32b663c6d6f7d5df64de73e19e02fcac789935f72565b8'}, {'path': 'README.md', 'size_bytes': 14967, 'sha256': 'a6812e0cca383af9f4c316f8abe43169ddda7230c2686ba314629e59a41a66b9'}], 'preprocessing': 'None: no shuffle, split, filtering, label conversion or token normalization.', 'parity_scope': 'Split sizes, metadata, all-row structure/IOB2 and ordered content hashes checked. Byte-for-byte equivalence to eriktks/conll2003 has not been established.', 'recorded_at_utc': '2026-11-01T12:48:22.774120+00:00'}, 'saved_data': {'path': '/root/cs5489_ner_local/data/conll2003_20261101T124819Z-467c97ad', 'read_back_verified': True, 'files': [{'path': 'dataset_dict.json', 'size_bytes': 43, 'sha256': '1239dbd4ca38311b03319b672c05a6fdaacc884b0034eed047659406c62eedd4'}, {'path': 'test/data-00000-of-00001.arrow', 'size_bytes': 1614264, 'sha256': 'c640eee74c9972843f89bbebd941fcb18556d81218ded7184f13c21b6bf216f6'}, {'path': 'test/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'test/state.json', 'size_bytes': 249, 'sha256': '6c9f6a869816eade461e5222e39056c0280aa74bf7793540b8f048f918c0533e'}, {'path': 'train/data-00000-of-00001.arrow', 'size_bytes': 7056024, 'sha256': 'd62e5b846961b150ea80d24d2ee411f1eb63365b6aa9d91631057a61246b755f'}, {'path': 'train/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'train/state.json', 'size_bytes': 250, 'sha256': '98c253b1e94a8d35ffa5b792381ecece2b6961d758c0e9a21e87c37c810352ef'}, {'path': 'validation/data-00000-of-00001.arrow', 'size_bytes': 1769816, 'sha256': 'ab07b4269a275b8a87b240b9169e245a647ab95353027f417118f321d16d5ee8'}, {'path': 'validation/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'validation/state.json', 'size_bytes': 255, 'sha256': 'b7d37bba572ad72b34ee9b39171d3e8d64d9b2789741ab05dbb810b47ff2f93d'}]}, 'audit': {'splits': {'train': {'rows': 14041, 'tokens': 203621, 'ordered_core_sha256': 'aa1208d2f06bc11610d0d9527af175e282ca0f4092a06a5a9381240e1dfece6a', 'id_order_verified': True, 'document_order_verified': True, 'documents': 946}, 'validation': {'rows': 3250, 'tokens': 51362, 'ordered_core_sha256': '91c3c12a9140e03ca679f24dc0437cdbae033babea946388677e46f9a52a78c2', 'id_order_verified': True, 'document_order_verified': True, 'documents': 216}, 'test': {'rows': 3453, 'tokens': 46435, 'ordered_core_sha256': 'bbdf60c2c1c3fbd1e7cb6a865d9d4dfd9df0110dff115de49f972175df7d3882', 'id_order_verified': True, 'document_order_verified': True, 'documents': 231}}, 'label_names': {'pos_tags': ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], 'chunk_tags': ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], 'ner_tags': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']}, 'iob2_violations': 0, 'hash_format': 'core-fields-decoded-tags-jsonl-utf8-v1'}, 'recovery': 'redownloaded-pinned-step1-source', 'run_id': '20261101T124819Z-467c97ad', 'acceptance': {'date': '2026-11-01', 'evidence': 'NER_step2_20261101T124840Z_e05b4ba0.zip', 'archive_sha256': '4b77cf2de940fb7b51a2a91fb757a33cc0698a3ea26ee655849e48659afd1e55'}}
for name, source in SOURCE_BUNDLE.items():
    (PROJECT_ROOT / 'src' / name).write_text(source, encoding='utf-8')

def load_project_module(name):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / 'src' / (name + '.py'))
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

backup = load_project_module('local_backup')
data_tools = load_project_module('step1_data')
eda_tools = load_project_module('step2_eda')
alignment = load_project_module('step3_alignment')
data_tools.write_json(PROJECT_ROOT / 'config' / 'step3_accepted_data_reference.json', ACCEPTED_DATA)
data_tools.write_json(PROJECT_ROOT / 'config' / 'preprocessing_policy.json', alignment.POLICY)
print('Tools and accepted Step 2 data reference loaded.')
print('Fixed max_length:', alignment.MAX_LENGTH, '(including special tokens)')


## 3.3 核对数据并建立标签映射
验证来源文件、顺序、IOB2 和三个划分的元数据。优先读取第 2 步保留的本地数据；缺失时自动恢复同一来源。
不更改词、大小写、标签、样本顺序或划分。


In [ ]:
STEP3_COMPLETE = DATA_READY = TOKENIZERS_READY = ALIGNMENT_VERIFIED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
raw_datasets, DATA_RECORD = eda_tools.ensure_dataset(PROJECT_ROOT, ACCEPTED_DATA, RUN_ID)
LABEL_NAMES = DATA_RECORD['audit']['label_names']['ner_tags']
LABEL2ID = {label:index for index,label in enumerate(LABEL_NAMES)}
ID2LABEL = {index:label for label,index in LABEL2ID.items()}
for split, dataset in raw_datasets.items():
    if dataset.features['ner_tags'].feature.names != LABEL_NAMES:
        raise ValueError('Label metadata differs across splits: ' + split)
LABEL_MAPPING = dict(label_names=LABEL_NAMES, label2id=LABEL2ID, id2label=ID2LABEL, scheme='IOB2')
data_tools.write_json(PROJECT_ROOT / 'config' / 'label_mapping.json', LABEL_MAPPING)
DATA_READY = True
print('Data verified:', DATA_RECORD['recovery'])
print('Rows:', {s:r['rows'] for s,r in DATA_RECORD['audit']['splits'].items()})
print('Label mapping:', LABEL2ID)


## 3.4 固定分词器与合成样例检查
下载固定 commit 的分词器配置、词表和 tokenizer.json，并保存每个文件的哈希。
RoBERTa 使用 `add_prefix_space=True`，支持已按词拆开的输入。
合成长句强制使用 64 的长度上限，检查跨段实体、首子词、特殊符号、补齐和原句重建；正式数据仍用 256。


In [ ]:
STEP3_COMPLETE = TOKENIZERS_READY = ALIGNMENT_VERIFIED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not DATA_READY:
    raise RuntimeError('Complete 3.3 first.')
TOKENIZERS, TOKENIZER_SOURCES, SYNTHETIC_CHECKS = {}, {}, {}
for key in alignment.TOKENIZER_SPECS:
    tokenizer, source = alignment.load_tokenizer(key, PROJECT_ROOT)
    checks = alignment.synthetic_alignment_check(tokenizer, LABEL_NAMES)
    TOKENIZERS[key], TOKENIZER_SOURCES[key], SYNTHETIC_CHECKS[key] = tokenizer, source, checks
    print(key, source['revision'], 'synthetic checks: PASS')
TOKENIZERS_READY = True
data_tools.write_json(RESULTS_DIR / 'synthetic_checks.json', SYNTHETIC_CHECKS)


## 3.5 对齐所有词、保存并读回核验
每个分词器处理 train / validation / test 三个划分。统计超过 256 的原句数量及长度分布。
本节可能运行数分钟，会每处理 5,000 句输出进度。

验收要求：原词与监督位置一一对应、无丢词或重复词、所有标签能还原到原句、每个片段不超过 256、
补齐标签正确、保存再读回后的有序内容哈希一致。
下面显示 6 行汇总和训练集对齐样例；测试集只做固定规则的转换与完整性核验，不据此选择配置。


In [ ]:
from IPython.display import display

STEP3_COMPLETE = ALIGNMENT_VERIFIED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not TOKENIZERS_READY:
    raise RuntimeError('Complete 3.4 first.')
PREPARATION = {}
for key, tokenizer in TOKENIZERS.items():
    print('Preparing:', key)
    PREPARATION[key] = alignment.prepare_tokenizer_dataset(
        raw_datasets, tokenizer, LABEL_NAMES, PREPROCESSED_DIR / key, key)

SUMMARIES = [row for record in PREPARATION.values() for row in record['summaries']]
if len(SUMMARIES) != 6 or not all(r['read_back_verified'] for r in PREPARATION.values()):
    raise RuntimeError('Expected both tokenizers and all three splits to pass read-back checks.')
SUMMARY = pd.DataFrame(SUMMARIES)
TABLES = {
    'alignment_summary': SUMMARY,
    'tokenized_length_histograms': pd.DataFrame([r for v in PREPARATION.values() for r in v['length_histograms']]),
    'overflow_sentences': pd.DataFrame([r for v in PREPARATION.values() for r in v['overflow']],
        columns=['tokenizer','split','sentence_index','sentence_id','words','full_length','chunks','multi_piece_words','unknown_pieces']),
    'alignment_examples': pd.DataFrame([r for v in PREPARATION.values() for r in v['alignment_examples']]),
}
for name, table in TABLES.items():
    table.to_csv(RESULTS_DIR / (name + '.csv'), index=False, encoding='utf-8')
REPORT = dict(run_id=RUN_ID, notebook_revision=NOTEBOOK_REVISION, seed=SEED,
    policy=alignment.POLICY, label_mapping=LABEL_MAPPING, accepted_reference=ACCEPTED_DATA['acceptance'],
    source=DATA_RECORD['source'], ordered_data_audit=DATA_RECORD['audit'],
    tokenizer_sources=TOKENIZER_SOURCES, synthetic_checks=SYNTHETIC_CHECKS, preparation=PREPARATION)
data_tools.write_json(RESULTS_DIR / 'preprocessing_report.json', REPORT)
data_tools.write_json(PROJECT_ROOT / 'config' / 'preprocessing_latest.json', {
    'run_id':RUN_ID, 'results_dir':str(RESULTS_DIR), 'policy':alignment.POLICY,
    'label_mapping':LABEL_MAPPING, 'tokenizer_sources':TOKENIZER_SOURCES,
    'datasets':{key:{k:record[k] for k in ('path','summaries','saved_files','read_back_verified')}
                for key,record in PREPARATION.items()},
    'verified':True, 'computer_download_confirmed':False,
})
(RESULTS_DIR / 'README.md').write_text(
    '# Step 3 preprocessing\n\n' + '\n'.join('- '+str(v) for v in alignment.POLICY.values()) +
    '\n\nThe maximum length was fixed before tokenized test lengths were observed. '
    'Test transformations are integrity checks only, with no parameter selection or scoring. '
    'First-subword supervision is verified for every original word. '
    'Gold-label round trips are alignment checks, not model accuracy measurements. '
    'Chunking can reduce cross-boundary context; I-tags at a chunk start are preserved. '
    'Future metrics must reconstruct whole sentences before entity extraction. '
    'Prediction reconstruction uses word_ids and does not inspect gold labels. '
    'Arrow files and saved tokenizers stay under data/ and are excluded from the ZIP. '
    'If the runtime is lost, rerun Step 3 with the same pinned sources. '
    'Retain earlier ZIPs and download the notebook separately.\n', encoding='utf-8')
ALIGNMENT_VERIFIED = True
display(SUMMARY[['tokenizer','split','sentences','chunks','original_words','supervised_words',
    'over_max_length_sentences','max_full_length','max_chunk_length','dropped_words','duplicated_words',
    'gold_roundtrip_passed','padding_labels_verified']])
for key in TOKENIZERS:
    print('Training alignment example:', key)
    display(TABLES['alignment_examples'].query('tokenizer == @key').head(24))
print('ALL ALIGNMENT AND SAVE/READ-BACK CHECKS PASSED.')
print('Prepared data:', PREPROCESSED_DIR)
print('Results:', RESULTS_DIR)


## 3.6 生成并下载备份
本步 ZIP 含源码、固定版本、标签映射、规则、4 张表和校验记录；数据与分词器缓存可重建，不进入 ZIP。
ZIP 在运行时生成后会读回验证；下载成功仍需要你在电脑上确认。


In [ ]:
STEP3_COMPLETE = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not (DATA_READY and TOKENIZERS_READY and ALIGNMENT_VERIFIED):
    raise RuntimeError('Complete 3.3–3.5 first.')
ARTIFACTS = [{'path':p.relative_to(PROJECT_ROOT).as_posix(), 'size_bytes':p.stat().st_size,
              'sha256':backup.sha256_file(p)} for p in sorted(RESULTS_DIR.iterdir()) if p.is_file()]
RUNTIME_RECORD = dict(run_id=RUN_ID, notebook_revision=NOTEBOOK_REVISION, seed=SEED,
    python=sys.version, platform=platform.platform(),
    packages={name:metadata.version(name) for name in [*REQUIRED, 'pyarrow']},
    data_recovery=DATA_RECORD['recovery'], source_sha256s=DATA_RECORD['source'],
    source_code_sha256s={name:backup.sha256_file(PROJECT_ROOT / 'src' / name) for name in SOURCE_BUNDLE},
    artifacts=ARTIFACTS, alignment_verified=True)
data_tools.write_json(PROJECT_ROOT / 'logs' / f'step3_environment_{RUN_ID}.json', RUNTIME_RECORD)
BACKUP_RECEIPT = backup.create_backup(PROJECT_ROOT, EXPORT_DIR, step='step3',
                                    revision=NOTEBOOK_REVISION, include_checkpoints=False)
BACKUP_PATH = Path(BACKUP_RECEIPT['path'])
BACKUP_VERIFIED = BACKUP_RECEIPT['archive_verified']
print('ZIP READY — check the computer copy after downloading.')
print(json.dumps(BACKUP_RECEIPT, indent=2))


In [ ]:
if not BACKUP_VERIFIED or not BACKUP_PATH.is_file():
    raise RuntimeError('Generate the verified ZIP first.')
try:
    from google.colab import files
    files.download(str(BACKUP_PATH))
    print('Download requested:', BACKUP_PATH.name)
except Exception as exc:
    print('Automatic download did not finish:', type(exc).__name__, str(exc))
print('Files-panel fallback path:', BACKUP_PATH)
print('Also save this notebook with File > Download .ipynb.')


## 3.7 确认电脑备份并反馈
确认电脑上有同名且字节数一致的 ZIP 后，将 `BACKUP_DOWNLOADED` 改成 `True`，再执行本节。
发送最后输出，并把下载的 ZIP 放进电脑的 `CS 5489/NER` 文件夹，方便复核。
如遇报错，反馈出错小节和错误文本；不要跳过检查。完整重跑从 3.2 开始，使用新 run_id。


In [ ]:
BACKUP_DOWNLOADED = False  # Set True only after checking the computer ZIP.
STEP3_COMPLETE = False
if not (DATA_READY and TOKENIZERS_READY and ALIGNMENT_VERIFIED and BACKUP_VERIFIED):
    raise RuntimeError('Complete 3.3–3.6 first.')
if not BACKUP_DOWNLOADED:
    print('WAITING: check the computer ZIP, then set BACKUP_DOWNLOADED=True.')
else:
    backup.verify_backup(BACKUP_PATH)
    if backup.sha256_file(BACKUP_PATH) != BACKUP_RECEIPT['sha256']:
        raise RuntimeError('The ZIP changed after its receipt was generated.')
    STEP3_COMPLETE = True
    print('STEP 3 COMPLETE: alignment and ZIP verified; computer download confirmed by the user.')
    print('Labels:', LABEL_NAMES)
    print('Max length:', alignment.MAX_LENGTH, '(including special tokens)')
    for key, record in PREPARATION.items():
        print(key, 'revision:', TOKENIZER_SOURCES[key]['revision'])
        print('  Original/supervised words:', {r['split']:[r['original_words'],r['supervised_words']] for r in record['summaries']})
        print('  Over-limit sentences/chunks:', {r['split']:[r['over_max_length_sentences'],r['chunks']] for r in record['summaries']})
    print('Dropped words: 0; duplicated words: 0; gold round trip: PASS; padding: PASS; saved read-back: PASS')
    print('Tables:', len(TABLES))
    print('Results:', RESULTS_DIR)
    print('Prepared data:', PREPROCESSED_DIR)
    print('Backup:', BACKUP_RECEIPT['name'])
    print('Bytes:', BACKUP_RECEIPT['size_bytes'])
    print('SHA-256:', BACKUP_RECEIPT['sha256'])
    print('Send this output before Step 4.')
